# Análisis Exploratorio — Propiedades Medellín

In [116]:
import pandas as pd

In [117]:
#csv_path = "https://dwwzqwcoqlasgvpniwiu.supabase.co/storage/v1/object/public/Data_bucket/aa/propiedades_medellin_raw.csv"
csv_path = "C:\\Users\\Acer\\Desktop\\prueba_inmobiliaria\\propiedades_medellin_raw.csv"
df = pd.read_csv(csv_path, encoding="utf-8-sig")

print(f"Filas    : {df.shape[0]:,}")
print(f"Columnas : {df.shape[1]}")
print("\nColumnas:", df.columns)


Filas    : 1,240
Columnas : 8

Columnas: Index(['id_propiedad', 'nombre_anuncio', 'precio_venta', 'metraje_m2',
       'estrato_socioeconomico', 'ubicacion', 'tipo_inmueble',
       'fecha_publicacion'],
      dtype='object')


In [118]:
df.head(15)

,id_propiedad,nombre_anuncio,precio_venta,metraje_m2,estrato_socioeconomico,ubicacion,tipo_inmueble,fecha_publicacion
0,681,Casa en Sabaneta,1989000000,249,2.0,SABANETA,Casa,"Oct 15, 2023"
1,377,Propiedad en Centro - Medellín,1514000000,359,6.0,Centro - Medellín,?,15/10/2023
2,1084,Casa en El Poblado,822000000,227,4.0,El Poblado,Casa,2023.10.15
3,948,Casa en El Poblado,828000000,44,2.0,El Poblado,Casa,15/10/2023
4,1017,None en Robledo,1226000000,353,4.0,ROBLEDO,NaN,15/10/2023
5,84,None en Centro,2009000000,269,2.0,Centro,NaN,2023-10-15
6,356,Casa en Centro,2464000000,367,2.0,Centro,Casa,2023-10-15
7,752,Apartestudio en Laureles - Estadio,1922000000,403,2.0,Laureles - Estadio,Apartestudio,15/10/2023
8,1147,Propiedad en Robledo,$ 2.441.000.000,327,2.0,Robledo,?,"Oct 15, 2023"
9,780,Apartestudio en Envigado,$ 2.375.000.000,0,5.0,ENVIGADO,Apartestudio,2023.10.15


**Observación:** aparecen literales `None` en `nombre_anuncio`, precios con formato `$ 2.441.000.000` y ubicaciones en distintos formatos. Esto indica que el ETL debe normalizar placeholders (`None`, `?`) y limpiar formatos numéricos.

In [119]:
df.dtypes

id_propiedad                int64
nombre_anuncio             object
precio_venta               object
metraje_m2                  int64
estrato_socioeconomico    float64
ubicacion                  object
tipo_inmueble              object
fecha_publicacion          object
dtype: object

In [120]:
df.isnull().sum()

id_propiedad                0
nombre_anuncio              0
precio_venta                0
metraje_m2                  0
estrato_socioeconomico    127
ubicacion                   0
tipo_inmueble             124
fecha_publicacion           0
dtype: int64

In [121]:
print("filas duplicadas",df.duplicated().sum())
print("filas con id duplicado",df.duplicated(subset=["id_propiedad"]).sum())

filas duplicadas 40
filas con id duplicado 40


**Observación:** se detectaron `id_propiedad` duplicados

In [122]:
ubicaciones = df["ubicacion"].value_counts(dropna=False)
print(f"Valores unicos: {df['ubicacion'].nunique(dropna=False)}")
print()
print(ubicaciones.to_string())

Valores unicos: 19

ubicacion
Laureles              81
laureles              78
BELEN                 76
ROBLEDO               74
ENVIGADO              71
Centro - Medellín     67
Belén                 65
poblado               65
EL HUECO              65
Centro                64
El Poblado            64
SABANETA              63
Envigado              62
Laureles - Estadio    61
Robledo               61
EL POBLADO            59
Belen                 58
Sabaneta              56
POBLADO               50


**Observación:** 19 variantes para 8 zonas reales — diferencias de casing, tildes y sufijos (`El Poblado`, `POBLADO`, `poblado`, `Laureles - Estadio`)

In [123]:
# tipo_inmueble — cuántos inválidos hay y qué valores exactos
print(df["tipo_inmueble"].value_counts(dropna=False).to_string())

tipo_inmueble
Finca           254
Apartamento     251
Casa            249
Apartestudio    241
NaN             124
?               121


**Observación:** existen valores ? y celdas vacías. Se mapean a `Sin clasificar` en la tabla maestra. No son recuperables desde `nombre_anuncio` — cuando el tipo es `?` el nombre dice `Propiedad en...`; cuando está vacío dice `None en...`.

In [124]:
print("Valores únicos de estrato_socioeconomico:")
print(df["estrato_socioeconomico"].value_counts(dropna=False).to_string())
print(f"\nTipo pandas detectado: {df['estrato_socioeconomico'].dtype}")
print(f"Nulos: {df['estrato_socioeconomico'].isna().sum()}")


Valores únicos de estrato_socioeconomico:
estrato_socioeconomico
2.0    239
6.0    229
3.0    218
5.0    214
4.0    213
NaN    127

Tipo pandas detectado: float64
Nulos: 127


**Observación:** tipo float64 con valores como `2.0`, `3.0`. Se convierte a `int` y se mapea a tabla maestra `estrato`. Los nulos se asignan al registro `Sin estrato`.

In [125]:
print("Muestra de valores únicos de fecha_publicacion:")
print(df["fecha_publicacion"].value_counts(dropna=False).head(20).to_string())

Muestra de valores únicos de fecha_publicacion:
fecha_publicacion
15/10/2023      322
2023-10-15      320
2023.10.15      310
Oct 15, 2023    288


**Observación:** se detectaron múltiples formatos (ej. Oct 15, 2023, 15/10/2023, 2023.10.15).

In [126]:
# metraje_m2 — valores inválidos (cero o negativos)
invalidos = df[df["metraje_m2"] <= 0]["metraje_m2"].value_counts()
print(f"Registros con metraje <= 0: {(df['metraje_m2'] <= 0).sum()}")
print(invalidos.to_string())

Registros con metraje <= 0: 39
metraje_m2
-50    23
 0     16


**Observación (metraje):** hay valores `0`, `1` y negativos (`-50`) que no tienen sentido físico; en ETL marcarlos como NULL y, si es posible, consultar la fuente para corregirlos.

### nombre_anuncio — ¿es un campo calculado?`

**Hipótesis:** el nombre es siempre la concatenación de 
`tipo_inmueble + " en " + ubicacion`.

In [128]:
def normalizar(texto):
    """Lowercase + strip + colapsa espacios."""
    return re.sub(r"\s+", " ", str(texto).strip().lower())

In [129]:
df["nombre_original_norm"] = df["nombre_anuncio"].apply(normalizar)

In [130]:
def reconstruir_nombre(row):
    tipo_raw = row["tipo_inmueble"]
    ubic = normalizar(str(row["ubicacion"]).strip())

    if pd.isna(tipo_raw) or str(tipo_raw).strip() == "":
        return "none en " + ubic
    
    tipo = normalizar(str(tipo_raw).strip())
    
    if tipo == "?":
        return "propiedad en " + ubic
    
    return tipo + " en " + ubic

In [131]:

df["nombre_reconstruido"] = df.apply(reconstruir_nombre, axis=1)

# Paso 3: comparar fila a fila
df[["nombre_anuncio", "nombre_original_norm", "nombre_reconstruido"]].head(8)

,nombre_anuncio,nombre_original_norm,nombre_reconstruido
0,Casa en Sabaneta,casa en sabaneta,casa en sabaneta
1,Propiedad en Centro - Medellín,propiedad en centro - medellín,propiedad en centro - medellín
2,Casa en El Poblado,casa en el poblado,casa en el poblado
3,Casa en El Poblado,casa en el poblado,casa en el poblado
4,None en Robledo,none en robledo,none en robledo
5,None en Centro,none en centro,none en centro
6,Casa en Centro,casa en centro,casa en centro
7,Apartestudio en Laureles - Estadio,apartestudio en laureles - estadio,apartestudio en laureles - estadio


In [132]:
coincide = (df["nombre_original_norm"] == df["nombre_reconstruido"]).sum()
total = len(df)

print(f"Coincidencia: {coincide}/{total} ({coincide/total*100:.1f}%)")


Coincidencia: 1240/1240 (100.0%)


**Hipotesis confirmada — 1240/1240 (100%)**

nombre_anuncio es un campo derivado: siempre es tipo_inmueble + " en " + ubicacion.

No se necesita en la tabla `propiedad`. Se puede reconstruir como vista
